Extract Data and Create DataFrame

In [0]:
import requests
from pyspark.sql import SparkSession

In [0]:
url = "https://www.freetogame.com/api/games"
resposes = requests.get(url)
games_data= resposes.json()
df = spark.createDataFrame(games_data)

In [0]:
display(df)

developer,freetogame_profile_url,game_url,genre,id,platform,publisher,release_date,short_description,thumbnail,title
Blizzard Entertainment,https://www.freetogame.com/overwatch,https://www.freetogame.com/open/overwatch,Shooter,540,PC (Windows),Activision Blizzard,2022-10-04,A hero-focused first-person team shooter from Blizzard Entertainment.,https://www.freetogame.com/g/540/thumbnail.jpg,Overwatch
"KRAFTON, Inc.",https://www.freetogame.com/pubg,https://www.freetogame.com/open/pubg,Shooter,516,PC (Windows),"KRAFTON, Inc.",2022-01-12,Get into the action in one of the longest running battle royale games PUBG Battlegrounds.,https://www.freetogame.com/g/516/thumbnail.jpg,PUBG: BATTLEGROUNDS
Darkflow Software,https://www.freetogame.com/enlisted,https://www.freetogame.com/open/enlisted,Shooter,508,PC (Windows),Gaijin Entertainment,2021-04-08,Get ready to command your own World War II military squad in Gaijin and Darkflow Software’s MMO squad-based shooter Enlisted.,https://www.freetogame.com/g/508/thumbnail.jpg,Enlisted
Wargaming Group Limited,https://www.freetogame.com/world-of-tanks-heat,https://www.freetogame.com/open/world-of-tanks-heat,Shooter,644,PC (Windows),Wargaming Group Limited,2026-05-26,A free-to-play hero-based shooter set in the World of Tanks universe.,https://www.freetogame.com/g/644/thumbnail.jpg,World of Tanks: HEAT
Hotta Studio,https://www.freetogame.com/nte,https://www.freetogame.com/open/nte,RPG,642,PC (Windows),Perfect World Entertainment,2026-04-29,Neverness to Everness is a gacha game that blends some GTA-style play with team-based combat.,https://www.freetogame.com/g/642/thumbnail.jpg,Neverness to Everness
NCSoft,https://www.freetogame.com/throne-and-liberty,https://www.freetogame.com/open/throne-and-liberty,MMORPG,590,PC (Windows),Amazon Games,2024-10-01,A free-to-play multi-platorm MMORPG from NCSoft and Amazon Games.,https://www.freetogame.com/g/590/thumbnail.jpg,Throne And Liberty
Mediatonic,https://www.freetogame.com/fall-guys,https://www.freetogame.com/open/fall-guys,Battle Royale,523,PC (Windows),Mediatonic,2020-08-04,Play the most competitive massively multiplayer party royale game featuring beans ever for free on a variety of platforms.,https://www.freetogame.com/g/523/thumbnail.jpg,Fall Guys
YOOZOO Games,https://www.freetogame.com/game-of-thrones-winter-is-coming,https://www.freetogame.com/open/game-of-thrones-winter-is-coming,Strategy,340,Web Browser,GTArcade,2019-11-14,A free-to-play browser-based RTS based on the George R.R. Martin novels and popular HBO series.,https://www.freetogame.com/g/340/thumbnail.jpg,Game Of Thrones Winter Is Coming
InnoGames,https://www.freetogame.com/elvenar,https://www.freetogame.com/open/elvenar,Strategy,347,Web Browser,InnoGames,2015-04-08,A browser based city-building strategy MMO set in the fantasy world of Elvenar.,https://www.freetogame.com/g/347/thumbnail.jpg,Elvenar
Cryptic Studios,https://www.freetogame.com/neverwinter,https://www.freetogame.com/open/neverwinter,MMORPG,11,PC (Windows),Perfect World Entertainment,2013-12-06,A free-to-play 3D action MMORPG based on the acclaimed Dungeons & Dragons fantasy roleplaying game.,https://www.freetogame.com/g/11/thumbnail.jpg,Neverwinter


In [0]:
df.printSchema()
print("Total Records", df.count())

root
 |-- developer: string (nullable = true)
 |-- freetogame_profile_url: string (nullable = true)
 |-- game_url: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- id: long (nullable = true)
 |-- platform: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- thumbnail: string (nullable = true)
 |-- title: string (nullable = true)

Total Records 413


Clean the Data

In [0]:
from pyspark.sql.functions import col,trim

In [0]:
for c in df.columns:
    print(c, df.filter(col (c).isNull()).count())

developer 0
freetogame_profile_url 0
game_url 0
genre 0
id 0
platform 0
publisher 0
release_date 0
short_description 0
thumbnail 0
title 0


In [0]:
from pyspark.sql.functions import trim
df_clean = (
    df
    .withColumn("title", trim(col("title")))
    .withColumn("genre", trim(col("genre")))
    .withColumn("platform", trim(col("platform")))
    .withColumn("publisher", trim(col("publisher")))
    .withColumn("developer", trim(col("developer")))
)

In [0]:
df_clean = df_clean.dropDuplicates(["id"])

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-----------------+
|catalog          |
+-----------------+
|dataset          |
|dbacademy        |
|ecommerce        |
|ecommerce_catalog|
|ecommercejune2026|
|main             |
|samples          |
|system           |
|tmdbapi          |
|workspace        |
+-----------------+



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dataset.freetogame")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS in dataset").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|default           |
|freetogame        |
|information_schema|
+------------------+



In [0]:
df_clean.write.mode("overwrite").saveAsTable("dataset.freetogame.games")

In [0]:
spark.sql("select * from dataset.freetogame.games limit 10").show()

+-------------------+----------------------+--------------------+----------+---+------------+--------------------+------------+--------------------+--------------------+-------------------+
|          developer|freetogame_profile_url|            game_url|     genre| id|    platform|           publisher|release_date|   short_description|           thumbnail|              title|
+-------------------+----------------------+--------------------+----------+---+------------+--------------------+------------+--------------------+--------------------+-------------------+
|          InnoGames|  https://www.freet...|https://www.freet...|  Strategy|347| Web Browser|           InnoGames|  2015-04-08|A browser based c...|https://www.freet...|            Elvenar|
|      Smilegate RPG|  https://www.freet...|https://www.freet...|      ARPG|517|PC (Windows)|        Amazon Games|  2022-02-11|Smilegate’s free-...|https://www.freet...|           Lost Ark|
|Battlefield Studios|  https://www.freet...|https:

In [0]:
spark.sql("SELECT count(*) As Total_count FROM dataset.freetogame.games").show()

+-----------+
|Total_count|
+-----------+
|        413|
+-----------+

